# RAG Agentik: Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari4/10-agentic-rag-latihan.ipynb)

Ini notebook **latihan** bab ini di Google Colab. Semua latihan (2 latihan) ada di satu notebook, dikerjakan berurutan dari atas ke bawah; setiap latihan diawali judul `Latihan N`. Solusi lengkap yang sudah terisi dan tinggal dijalankan ada di [notebook solusi](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari4/10-agentic-rag-solusi.ipynb).

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-agent-migas/blob/main/hari4/10-agentic-rag/10-agentic-rag.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Latihan 1: Asisten SOP dengan Uji Otomatis
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Asisten dokumen untuk operator lapangan, dengan enam uji yang diperiksa DI KODE:

1. Pengetahuan umum -> menjawab langsung, TIDAK mencari
2. Pertanyaan SOP -> mencari, menjawab dengan angka yang benar, mengutip SOP yang benar
3. Konflik revisi -> memakai Rev 3 (10 ppm), tidak memakai Rev 2 (20 ppm)
4. Kode alarm alat -> menemukan manual VSD dan menjelaskan kode F07
5. Tak terjawab -> menolak dengan kalimat standar
6. Dokumen terlarang -> menolak (operator tidak boleh membaca laporan investigasi HSE)

In [ ]:
# Langkah 1 - Menyiapkan agen
import os
import sys

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from migas_shared.config import ROOT
from migas_shared.rag_tools import KonteksPengguna

sys.path.insert(0, str(ROOT / "hari4" / "10-agentic-rag" / "code"))  # agar rag_agent.py ditemukan
from rag_agent import HasilRAG, bangun_agen_rag, tanya

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)
# TODO(B10.1): buat agen RAG dengan bangun_agen_rag(model)
agent = None  # TODO: ganti dengan implementasi Anda
operator = KonteksPengguna(role="operator", area="Seruni", user_id="OP-114")

## Langkah 2 - Fungsi pemeriksa
`periksa` mencetak LOLOS atau GAGAL beserta rincian, lalu mengembalikan hasilnya.

In [ ]:
def periksa(nama: str, ok: bool, rincian: str) -> bool:
    print(f"{'LOLOS' if ok else 'GAGAL'}  {nama}")
    print(f"        {rincian}")
    return ok


hasil_uji: list[bool] = []

## Langkah 3 - Uji 1: pengetahuan umum

In [ ]:
r1: HasilRAG = tanya(agent, "Apa ibu kota Prancis?", operator)
hasil_uji.append(periksa("1. Pengetahuan umum dijawab langsung", not r1.mencari and "paris" in r1.jawaban.lower(),
                         f"mencari={r1.mencari}, jawaban: {r1.jawaban[:80]}"))

## Langkah 4 - Uji 2: pertanyaan SOP

In [ ]:
r2 = tanya(agent, "Berapa nilai IDLH untuk gas H2S?", operator)
# TODO(B10.2): uji 2: tambahkan hasil periksa(...) yang meminta agen mencari, jawaban memuat '100 ppm', sitasi memuat SOP-HSE-001, dan tidak ada sitasi tidak valid
pass  # TODO: ganti dengan implementasi Anda

## Langkah 5 - Uji 3: konflik revisi

In [ ]:
r3 = tanya(agent, "Pada konsentrasi H2S berapa pekerja wajib menghentikan pekerjaan dan evakuasi?", operator)
# TODO(B10.3): uji 3: jawaban memuat '10 ppm' dan tidak ada revisi yang diambil berstatus superseded
pass  # TODO: ganti dengan implementasi Anda

## Langkah 6 - Uji 4: kode alarm alat

In [ ]:
r4 = tanya(agent, "Apa arti alarm F07 pada VSD FD-500 dan apakah unit boleh langsung di-restart?", operator)
hasil_uji.append(periksa("4. Kode alarm alat: manual VSD ditemukan",
                         "MAN-VSD-FD500" in r4.diambil and ("ground fault" in r4.jawaban.lower() or "f07" in r4.jawaban.lower()),
                         f"diambil={r4.diambil}"))

## Langkah 7 - Uji 5: pertanyaan tak terjawab

In [ ]:
r5 = tanya(agent, "Siapa direktur utama PT Hulu Energi Nusantara?", operator)
hasil_uji.append(periksa("5. Pertanyaan tak terjawab ditolak", r5.menolak, f"jawaban: {r5.jawaban[:90]}"))

## Langkah 8 - Uji 6: dokumen terlarang

In [ ]:
r6 = tanya(agent, "Apa akar masalah insiden paparan H2S di SR-027 pada Juli 2026?", operator)
# TODO(B10.4): uji 6: INV-2026-011 tidak ada di r6.diambil dan agen menolak (r6.menolak)
pass  # TODO: ganti dengan implementasi Anda

## Langkah 9 - Rekap

In [ ]:
print(f"Hasil: {sum(hasil_uji)}/{len(hasil_uji)} uji lolos")
print("(Uji berbasis model dapat sesekali gagal. Jalankan ulang dan analisis mengapa: salah cari, salah revisi, atau salah menolak?)")

## Latihan Bonus: RAG Percakapan
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Menambahkan memori (bab 5) ke agen RAG agar pengguna bisa mengajukan follow-up question yang merujuk
jawaban sebelumnya, tanpa mengulang konteks:

- "Berapa lama fire watch harus berjaga setelah pekerjaan panas?" -> 30 menit
- "Dan bagaimana dengan batas LEL-nya?" -> (masih soal pekerjaan panas)
- "Dokumen mana yang tadi kamu rujuk?" -> menjawab dari riwayat, tanpa mencari

In [ ]:
# Langkah 1 - Menyiapkan agen dengan memori percakapan
import os
import re
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from migas_shared.config import ROOT
from migas_shared.rag_tools import KonteksPengguna, cari_dokumen

sys.path.insert(0, str(ROOT / "hari4" / "10-agentic-rag" / "code"))  # agar rag_agent.py ditemukan
from rag_agent import PROMPT_RAG

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)
# TODO(B10.5): buat agen RAG percakapan: tool cari_dokumen, PROMPT_RAG ditambah izin pertanyaan lanjutan, context_schema=KonteksPengguna, dan checkpointer=InMemorySaver()
agent = None  # TODO: ganti dengan implementasi Anda
konteks = KonteksPengguna(role="operator", area="Seruni", user_id="OP-114")
config = {"configurable": {"thread_id": "sesi-operator-114"}}

## Langkah 2 - Fungsi satu giliran
`giliran` mengirim satu pertanyaan ke utas yang sama dan menghitung berapa kali agen mencari pada giliran itu.

In [ ]:
jumlah_pencarian = []


def giliran(pertanyaan: str) -> None:
    print(f"Pengguna: {pertanyaan}")
    sebelum = len(agent.get_state(config).values.get("messages", [])) if jumlah_pencarian else 0
    # TODO(B10.6): panggil agent.invoke dengan HumanMessage, config thread yang sama, dan context=konteks
    response = None  # TODO: ganti dengan implementasi Anda
    baru = response["messages"][sebelum:]
    # TODO(B10.7): hitung berapa kali cari_dokumen dipanggil pada pesan baru (AIMessage dengan tool_calls bernama cari_dokumen)
    mencari = 0  # TODO: ganti dengan implementasi Anda
    jumlah_pencarian.append(mencari)
    print(f"   Pencarian pada giliran ini: {mencari}")
    print(f"Agen: {response['messages'][-1].content}")

## Langkah 3 - Giliran 1: pertanyaan awal

In [ ]:
giliran("Berapa lama fire watch harus tetap berjaga setelah pekerjaan panas selesai?")

## Langkah 4 - Giliran 2: pertanyaan lanjutan
Pertanyaan ini bergantung pada konteks giliran sebelumnya.

In [ ]:
giliran("Dan bagaimana dengan batas LEL yang disyaratkan?")

## Langkah 5 - Giliran 3: merujuk riwayat

In [ ]:
giliran("Dokumen apa yang tadi kamu rujuk untuk dua jawaban itu?")

## Langkah 6 - Pemeriksaan di kode

In [ ]:
status = agent.get_state(config)
print(f"Utas menyimpan {len(status.values['messages'])} pesan (pertanyaan, pencarian, hasil, dan jawaban).")
teks_akhir = str(status.values["messages"][-1].content)
ok_ingatan = bool(re.search(r"SOP-HSE-004|LL-2026-03", teks_akhir))
print(f"{'LOLOS' if ok_ingatan else 'GAGAL'}  pertanyaan ke-3 menyebut dokumen dari giliran sebelumnya (SOP-HSE-004)")
print(f"{'LOLOS' if jumlah_pencarian[0] >= 1 else 'GAGAL'}  pertanyaan pertama mencari di dokumen")